# 仮想水輸出が多い20傑

In [2]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ===== 設定 =====
VWT_DIR = Path(r"C:\修論研究\VWT_historical_data\VWT_npy")
ITEM_LIST_XLSX = r"C:\修論研究\VWT_historical_data\item_list.xlsx"
YEAR = 2016
TOP_N = 20

# ===== 2016年の作物別・世界総仮想水輸出量を集計 =====
pat = re.compile(rf"^VWT_(\d+)_{YEAR}\.npy$")
rows = []

for p in VWT_DIR.glob(f"VWT_*_{YEAR}.npy"):
    m = pat.match(p.name)
    if not m:
        continue
    crop_id = int(m.group(1))
    vwt_mat = np.load(p).astype(float)
    if vwt_mat.ndim != 2:
        continue

    # 世界総輸出量（行列全要素の合計）
    world_export_total = float(np.nansum(vwt_mat))
    rows.append(
        {
            "crop_id": crop_id,
            "world_export_total_m3_per_year": world_export_total,
        }
    )

top20 = (
    pd.DataFrame(rows)
    .sort_values("world_export_total_m3_per_year", ascending=False)
    .head(TOP_N)
    .reset_index(drop=True)
)

# ===== item_list.xlsx から Commodity name を付与 =====
item_df = pd.read_excel(ITEM_LIST_XLSX)

def find_col(cols, exact=None, contains=None):
    exact = exact or []
    contains = contains or []
    lowered = {str(c).strip().lower(): c for c in cols}
    for k in exact:
        if k.lower() in lowered:
            return lowered[k.lower()]
    for c in cols:
        low = str(c).strip().lower()
        if all(token in low for token in contains):
            return c
    return None

fao_col = find_col(
    item_df.columns,
    exact=["FAO code", "fao code", "FAO_code", "fao_code"],
    contains=["fao", "code"],
)
name_col = find_col(
    item_df.columns,
    exact=["Commodity name", "commodity name", "Item name", "item name"],
    contains=["commodity", "name"],
)

if fao_col is None or name_col is None:
    raise ValueError(f"item_list.xlsx の列を特定できません。columns={list(item_df.columns)}")

item_map = item_df[[fao_col, name_col]].copy()
item_map = item_map.rename(columns={fao_col: "crop_id", name_col: "commodity_name"})
item_map["crop_id"] = pd.to_numeric(item_map["crop_id"], errors="coerce").astype("Int64")
item_map = item_map.dropna(subset=["crop_id"]).copy()
item_map["crop_id"] = item_map["crop_id"].astype(int)
item_map = item_map.drop_duplicates(subset=["crop_id"], keep="first")

# ===== 結合して表示 =====
top20_named = top20.merge(item_map, on="crop_id", how="left")
top20_named = top20_named[["crop_id", "commodity_name", "world_export_total_m3_per_year"]]
top20_named.index = top20_named.index + 1
top20_named.index.name = "rank"

display(top20_named)
print("top20 crop_id:", top20_named["crop_id"].tolist())


,crop_id,commodity_name,world_export_total_m3_per_year
rank,,,
1,15,Wheat,2.213492e+11
2,236,Soybeans,2.195880e+11
3,257,Palm oil,1.672256e+11
4,56,Maize,1.070263e+11
5,238,Cake of Soybeans,9.880626e+10
6,666,Chocolate Prsnes,9.501316e+10
7,30,Rice - total (Rice milled equivalent),9.176010e+10
8,656,"Coffee, green",8.466443e+10
9,661,Cocoa beans,7.100909e+10


top20 crop_id: [15, 236, 257, 56, 238, 666, 30, 656, 661, 767, 162, 237, 270, 268, 164, 664, 44, 167, 261, 271]


# 各国の20傑作物の輸出入動向を調べ、グループ化（主成分分析）一人当たりの方がいいかな